# EPCLM-001 v0.3 controlled baseline retraining
Enable one GPU, Internet, and existing HF_TOKEN/GITHUB_TOKEN secrets. One Run All: report-return preflight → exact historical data gate → durable storage probe → T1/X3/X4 seed 91002 only → strict export/readback. Two GPU-hour training guard. Stops before any formal E1 benchmark.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path("/kaggle/working/epclm001-retrain-code")
OUTPUT = Path("/kaggle/working/epclm001-retrain-output")
CACHE = Path("/kaggle/working/epclm001-retrain-data")


def run(*args):
    subprocess.run([str(a) for a in args], check=True)


for label in ("HF_TOKEN", "GITHUB_TOKEN"):
    if not os.environ.get(label):
        try:
            from kaggle_secrets import UserSecretsClient

            os.environ[label] = UserSecretsClient().get_secret(label)
        except Exception as error:  # noqa: BLE001 -- external Kaggle secret-client boundary
            print(label + "_unavailable", type(error).__name__)
print({label + "_available": bool(os.environ.get(label)) for label in ("HF_TOKEN", "GITHUB_TOKEN")})
if not (ROOT / ".git").exists():
    run(
        "git",
        "clone",
        "--filter=blob:none",
        "--no-checkout",
        "https://github.com/ArcheLabs/mini-cells.git",
        ROOT,
    )
run("git", "-C", ROOT, "fetch", "--depth=1", "origin", "codex/epclm001-incremental")
run("git", "-C", ROOT, "sparse-checkout", "init", "--no-cone")
run(
    "git",
    "-C",
    ROOT,
    "sparse-checkout",
    "set",
    "--no-cone",
    "/pyproject.toml",
    "/src/",
    "/shared/",
    "/research/projects/cellular-language-models/",
    "/research/projects/continual-learning/context/",
    "/research/projects/cellular-computing/experiments/007-minicells-30m/artifacts/tokenizer.json",
)
run("git", "-C", ROOT, "checkout", "--detach", "FETCH_HEAD")
# Reuse installed dependencies; the worker needs no Transformers/Gradio/Diffusers.
# Do not downgrade unrelated Kaggle packages through the broad LM extra.
import importlib.util

requirements = {
    "datasets": "datasets>=2.18",
    "tokenizers": "tokenizers>=0.15",
    "huggingface_hub": "huggingface_hub>=0.23",
}
missing = [
    spec for module, spec in requirements.items() if importlib.util.find_spec(module) is None
]
if missing:
    run(sys.executable, "-m", "pip", "install", "-q", *missing)
run(sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", ROOT)
entry = (
    ROOT
    / "research/projects/cellular-language-models/experiments/epclm-001-incremental-execution/run.py"
)
try:
    run(
        sys.executable, entry, "retrain-run", "--root", OUTPUT, "--cache", CACHE, "--publish-github"
    )
finally:
    print("Save Version retains resume/inference weights and reports:", OUTPUT)
    print(
        "Weights are independently uploaded/read back before ASSET_AVAILABLE. No E1-C or 30M run."
    )
    p = OUTPUT / "TRAINING_MANIFEST.json"
    if p.exists():
        value = json.loads(p.read_text())
        print(
            {
                "status": value["status"],
                "models": value["models"],
                "gpu_seconds_charged": value["gpu_seconds_charged"],
            }
        )